# 07_01 Backpropagation by hand: where does a network's correction come from?

A network is wrong, and it has thousands of weights. Which ones should change, and by how much? The answer
is a number for every weight, its **gradient**: how much the error would rise if that weight rose a little.
In this notebook you compute gradients with the chain rule, by hand, for the network the book's chapter
draws, and watch PyTorch compute exactly the same numbers in one line.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-07-how-a-network-learns", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'nltk': 'nltk',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    import nltk
    for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'omw-1.4', 'averaged_perceptron_tagger_eng', 'maxent_ne_chunker_tab', 'words', 'reuters']:
        nltk.download(pkg, quiet=True)
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import torch
from nlpcheck import ask, guess, reveal, check_07_01, house_network

torch.set_num_threads(4)
print("PyTorch", torch.__version__)

## 1. Recall

**r1.** What does a sigmoid do to a very large positive input, say 10?
(a) returns 10, (b) returns almost exactly 1, and its slope there is almost 0, (c) returns 0

**r2.** How does the chain rule find the derivative of a quantity that depends on a weight only through
other quantities? (a) it adds the local derivatives, (b) it takes the largest one, (c) it multiplies the
local derivatives along the path

In [ ]:
ask("r1", "")
ask("r2", "")

## 2. The smallest possible network: two operations

The chapter's quiz: `x = -2`, `y = 5`, `z = -4`, `q = x + y`, `f = q * z`. Work out the gradient of `f`
with respect to each of `x`, `y` and `z` on paper first. Record it as a tuple, for example `(1, 2, 3)`.

In [ ]:
guess("quiz_grads", None)   # a tuple (df/dx, df/dy, df/dz)

In [ ]:
x = torch.tensor(-2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
z = torch.tensor(-4.0, requires_grad=True)
q = x + y
f = q * z
f.backward()          # walk back from f, filling .grad on every tensor that asked for it
print("f =", f.item(), "| df/dx =", x.grad.item(), "| df/dy =", y.grad.item(), "| df/dz =", z.grad.item())
reveal("quiz_grads", (x.grad.item(), y.grad.item(), z.grad.item()))

`(-4, -4, 3)`. The chain rule, one link at a time: `f = q * z`, so `df/dq = z = -4` and `df/dz = q = 3`.
And `q = x + y`, so `dq/dx = dq/dy = 1`. Multiply along each path: `df/dx = df/dq * dq/dx = -4 * 1`.

`requires_grad=True` asks PyTorch to **record** every operation on the tensor as it happens, building a
graph. `backward()` walks that graph from the output to the inputs, multiplying local derivatives on the
way. That is **automatic differentiation**, and it is how every modern framework trains a network: nobody
writes gradients by hand any more. You are about to, once, so you know what it is doing.

## 3. The book's network: four inputs, five hidden neurons, one output

The chapter's example predicts a house price from four numbers. Here is its first house, each input divided
by a round number so they are all of similar size, and a target price of 0.24 (in millions). The weights
start as random numbers between 0 and 1, from a fixed seed, so everyone sees the same ones.

In [ ]:
X, Y, W_ih, b_ih, W_ho, b_ho = house_network()
for p in (W_ih, b_ih, W_ho, b_ho):
    p.requires_grad_(True)
print("inputs X:", X.tolist(), " target Y:", Y.item())
print("shapes:", tuple(W_ih.shape), tuple(b_ih.shape), tuple(W_ho.shape), tuple(b_ho.shape))

Z1 = X @ W_ih + b_ih          # (1, 5): each hidden neuron's weighted sum
h = torch.sigmoid(Z1)          # (1, 5): the hidden layer's outputs
Z2 = h @ W_ho + b_ho          # (1, 1)
O = torch.sigmoid(Z2)          # the prediction
E = 0.5 * ((Y - O) ** 2).sum()  # the error, the book's (Y - O)^2 / 2
print(f"prediction O = {O.item():.4f}, error E = {E.item():.4f}")

The prediction is 0.957 against a target of 0.24: badly wrong. Now the gradients, first from PyTorch:

In [ ]:
E.backward()
print("dE/dW_ho from autograd:", W_ho.grad.flatten().tolist())

Now by hand, exactly as the chapter derives it. Follow the error back from the output:

- `dE/dO = O - Y` (the derivative of `(Y - O)^2 / 2`; the 2s cancel)
- `dO/dZ2 = O * (1 - O)` (the slope of the sigmoid, written in terms of its own output)
- `dZ2/dW_ho = h` (because `Z2 = h @ W_ho + b_ho`)

Multiplying the first two gives one number, the output layer's **delta**, which every weight into the output
shares. The worked example:

In [ ]:
with torch.no_grad():
    delta2 = (O - Y) * O * (1 - O)      # (1, 1): dE/dZ2
    dW_ho = h.T @ delta2                # (5, 1): each hidden output times the shared delta
    db_ho = delta2.sum(0)               # (1,):   the bias sees an input of 1
print("dE/dW_ho by hand:     ", dW_ho.flatten().tolist())
print("largest difference from autograd:", (dW_ho - W_ho.grad).abs().max().item())

A difference of zero, or something like `1e-9` (the last digits of floating-point arithmetic).

## 4. Your turn: one layer further back

The weights between the input and the hidden layer, `W_ih`, reach the error through two more links:

- `dZ2/dh = W_ho` (each hidden output reaches `Z2` through its weight)
- `dh/dZ1 = h * (1 - h)` (the hidden sigmoid's slope)
- `dZ1/dW_ih = X`

So the hidden layer's delta is `delta1 = (delta2 @ W_ho.T) * h * (1 - h)`, shape (1, 5), and then
`dW_ih = X.T @ delta1`, shape (4, 5), and `db_ih = delta1.sum(0)`. Fill in the two lines.

In [ ]:
with torch.no_grad():
    delta1 = None   # YOUR CODE HERE: (delta2 @ W_ho.T), times the hidden sigmoid's slope
    dW_ih = None    # YOUR CODE HERE: X.T @ delta1
    db_ih = delta1.sum(0) if delta1 is not None else None

if dW_ih is not None:
    print("largest difference from autograd for W_ih:", (dW_ih - W_ih.grad).abs().max().item())

In [ ]:
os.makedirs("out", exist_ok=True)
tolist = lambda t: t.tolist() if t is not None else None
json.dump({"dW_ho": tolist(dW_ho), "db_ho": tolist(db_ho), "dW_ih": tolist(dW_ih), "db_ih": tolist(db_ih)},
          open("out/07_01_gradients.json", "w"), indent=1)
check_07_01()

## 5. One step of gradient descent

Now use the gradients: `W_new = W_old - learning_rate * gradient`, for all four tensors. With a learning rate
of 1, how much will one step reduce the error of 0.257? (a) almost to zero, (b) by about half, (c) barely.

In [ ]:
guess("one_step", None)   # "a", "b" or "c" 

In [ ]:
with torch.no_grad():
    for p in (W_ih, b_ih, W_ho, b_ho):
        p -= 1.0 * p.grad
    O_new = torch.sigmoid(torch.sigmoid(X @ W_ih + b_ih) @ W_ho + b_ho)
    E_new = 0.5 * ((Y - O_new) ** 2).sum()
print(f"error before {E.item():.4f}, after one step {E_new.item():.4f}")
print(f"slope of the output sigmoid at O = {O.item():.3f}: O * (1 - O) = {(O * (1 - O)).item():.3f}")
reveal("one_step", "c")

Barely: from 0.257 to 0.254. The step is exactly as the chain rule said, and the chain rule said it was
small, because one of its links is small. The output neuron is **saturated**: its sigmoid sits at 0.957,
where the curve is almost flat, so `O * (1 - O)` is 0.04, and every gradient passing through it is
multiplied by 0.04. Hold on to that: a small factor, multiplied in again and again, is the whole story of
this lab's last notebook.

## 6. Exit ticket

**x1.** For `f = (x + y) * z` at `x = -2, y = 5, z = -4`, what is `(df/dx, df/dy, df/dz)`?
(a) (-3, 4, 4), (b) (-4, -4, 3), (c) (3, -4, -4)

In [ ]:
ask("x1", "")

Explain it back: why did a step with learning rate 1 barely change the error? One or two sentences.

*Your explanation:* 